# Lab | Data Cleaning and Formatting

In this lab, we will be working with the customer data from an insurance company, which can be found in the CSV file located at the following link: https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv


# Challenge 1: Data Cleaning and Formatting

## Exercise 1: Cleaning Column Names

To ensure consistency and ease of use, standardize the column names of the dataframe. Start by taking a first look at the dataframe and identifying any column names that need to be modified. Use appropriate naming conventions and make sure that column names are descriptive and informative.

*Hint*:
- *Column names should be in lower case*
- *White spaces in column names should be replaced by `_`*
- *`st` could be replaced for `state`*

In [18]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv"
df = pd.read_csv(url)

print("Original columns:")
print(list(df.columns))

Original columns:
['Customer', 'ST', 'GENDER', 'Education', 'Customer Lifetime Value', 'Income', 'Monthly Premium Auto', 'Number of Open Complaints', 'Policy Type', 'Vehicle Class', 'Total Claim Amount']


In [19]:
df.columns = (
    df.columns
      .str.strip()                
      .str.lower()                
      .str.replace(r"\s+", "_", regex=True)
)

In [20]:
df.rename(columns={"st": "state"}, inplace=True)

In [21]:
print("Cleaned columns:")
print(list(df.columns))

Cleaned columns:
['customer', 'state', 'gender', 'education', 'customer_lifetime_value', 'income', 'monthly_premium_auto', 'number_of_open_complaints', 'policy_type', 'vehicle_class', 'total_claim_amount']


## Exercise 2: Cleaning invalid Values

The dataset contains columns with inconsistent and incorrect values that could affect the accuracy of our analysis. Therefore, we need to clean these columns to ensure that they only contain valid data.

Note that this exercise will focus only on cleaning inconsistent values and will not involve handling null values (NaN or None).

*Hint*:
- *Gender column contains various inconsistent values such as "F", "M", "Femal", "Male", "female", which need to be standardized, for example, to "M" and "F".*
- *State abbreviations be can replaced with its full name, for example "AZ": "Arizona", "Cali": "California", "WA": "Washington"*
- *In education, "Bachelors" could be replaced by "Bachelor"*
- *In Customer Lifetime Value, delete the `%` character*
- *In vehicle class, "Sports Car", "Luxury SUV" and "Luxury Car" could be replaced by "Luxury"*

In [22]:
gender_norm = (
    df["gender"]
      .astype(str)
      .str.strip()
      .str.lower()
      .replace({
          "m": "M", "male": "M",
          "f": "F", "female": "F", "femal": "F"  # casos mal escritos
      })
)

In [23]:
state_map = {
    "az": "Arizona", "AZ": "Arizona",
    "cali": "California", "Cali": "California", "CA": "California",
    "wa": "Washington", "WA": "Washington"
}
df["state"] = (
    df["state"]
      .astype(str)
      .str.strip()
      .replace(state_map)
)

In [24]:
state_map = {
    "az": "Arizona", "AZ": "Arizona",
    "cali": "California", "Cali": "California", "CA": "California",
    "wa": "Washington", "WA": "Washington"
}
df["state"] = (
    df["state"]
      .astype(str)
      .str.strip()
      .replace(state_map)
)

In [25]:
df["customer_lifetime_value"] = (
    df["customer_lifetime_value"]
      .astype(str)
      .str.strip()
      .str.replace("%", "", regex=False)
)

In [26]:
df["vehicle_class"] = (
    df["vehicle_class"]
      .astype(str)
      .str.strip()
      .str.title()
      .replace({
          "Sports Car": "Luxury",
          "Luxury Suv": "Luxury",
          "Luxury Car": "Luxury"
      })
)

## Exercise 3: Formatting data types

The data types of many columns in the dataset appear to be incorrect. This could impact the accuracy of our analysis. To ensure accurate analysis, we need to correct the data types of these columns. Please update the data types of the columns as appropriate.

It is important to note that this exercise does not involve handling null values (NaN or None).

*Hint*:
- *Customer lifetime value should be numeric*
- *Number of open complaints has an incorrect format. Look at the different values it takes with `unique()` and take the middle value. As an example, 1/5/00 should be 5. Number of open complaints is a string - remember you can use `split()` to deal with it and take the number you need. Finally, since it should be numeric, cast the column to be in its proper type.*

In [27]:
if "Customer Lifetime Value" in df.columns:
    df["Customer Lifetime Value"] = (
        df["Customer Lifetime Value"]
          .astype(str)
          .str.replace("%", "", regex=False)
          .str.replace(",", "", regex=False)
    )
    df["Customer Lifetime Value"] = pd.to_numeric(df["Customer Lifetime Value"], errors="coerce")


In [28]:
col_noc = "Open Complaints"
if col_noc in df.columns:
    noc_str = df[col_noc].astype(str)
    df[col_noc] = noc_str.apply(
        lambda x: x.split("/")[1] if ("/" in x and len(x.split("/")) >= 2) else x
    )
    df[col_noc] = pd.to_numeric(df[col_noc], errors="coerce")

In [29]:
for col in ["Income", "Monthly Premium Auto", "Total Claim Amount"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
print(df.dtypes)

customer                      object
state                         object
gender                        object
education                     object
customer_lifetime_value       object
income                       float64
monthly_premium_auto         float64
number_of_open_complaints     object
policy_type                   object
vehicle_class                 object
total_claim_amount           float64
dtype: object


## Exercise 4: Dealing with Null values

Identify any columns with null or missing values. Identify how many null values each column has. You can use the `isnull()` function in pandas to find columns with null values.

Decide on a strategy for handling the null values. There are several options, including:

- Drop the rows or columns with null values
- Fill the null values with a specific value (such as the column mean or median for numerical variables, and mode for categorical variables)
- Fill the null values with the previous or next value in the column
- Fill the null values based on a more complex algorithm or model (note: we haven't covered this yet)

Implement your chosen strategy to handle the null values. You can use the `fillna()` function in pandas to fill null values or `dropna()` function to drop null values.

Verify that your strategy has successfully handled the null values. You can use the `isnull()` function again to check if there are still null values in the dataset.

Remember to document your process and explain your reasoning for choosing a particular strategy for handling null values.

After formatting data types, as a last step, convert all the numeric variables to integers.

In [30]:
print("Nulls per column (before):")
print(df.isna().sum().sort_values(ascending=False))

Nulls per column (before):
gender                       3054
customer                     2937
education                    2937
total_claim_amount           2937
income                       2937
monthly_premium_auto         2937
number_of_open_complaints    2937
policy_type                  2937
state                           0
customer_lifetime_value         0
vehicle_class                   0
dtype: int64


In [34]:
num_cols = df.select_dtypes(include="number").columns
cat_cols = df.select_dtypes(exclude="number").columns

In [36]:
df[num_cols] = df[num_cols].apply(lambda s: s.fillna(s.median()))

def fill_with_mode(s: pd.Series):
    m = s.mode(dropna=True)
    return s.fillna(m.iloc[0]) if not m.empty else s

df[cat_cols] = df[cat_cols].apply(fill_with_mode)

In [ ]:
print("\nNulls per column (after fill):")
print(df.isna().sum().sort_values(ascending=False))

In [ ]:
df[num_cols] = df[num_cols].round().astype("int64")

print("\nDtypes after int conversion:")
print(df.dtypes)

## Exercise 5: Dealing with duplicates

Use the `.duplicated()` method to identify any duplicate rows in the dataframe.

Decide on a strategy for handling the duplicates. Options include:
- Dropping all duplicate rows
- Keeping only the first occurrence of each duplicated row
- Keeping only the last occurrence of each duplicated row
- Dropping duplicates based on a subset of columns
- Dropping duplicates based on a specific column

Implement your chosen strategy using the `drop_duplicates()` function.

Verify that your strategy has successfully handled the duplicates by checking for duplicates again using `.duplicated()`.

Remember to document your process and explain your reasoning for choosing a particular strategy for handling duplicates.

Save the cleaned dataset to a new CSV file.

*Hint*: *after dropping duplicates, reset the index to ensure consistency*.

In [ ]:
print("Total duplicate rows:", df.duplicated().sum())
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
print("Total duplicate rows:", df.duplicated().sum())
print("Total duplicate rows:", df.duplicated().sum())

# Bonus: Challenge 2: creating functions on a separate `py` file

Put all the data cleaning and formatting steps into functions, and create a main function that performs all the cleaning and formatting.

Write these functions in separate .py file(s). By putting these steps into functions, we can make the code more modular and easier to maintain.

*Hint: autoreload module is a utility module in Python that allows you to automatically reload modules in the current session when changes are made to the source code. This can be useful in situations where you are actively developing code and want to see the effects of changes you make without having to constantly restart the Python interpreter or Jupyter Notebook kernel.*

In [ ]:
# Your code here

# Bonus: Challenge 3: Analyzing Clean and Formated Data

You have been tasked with analyzing the data to identify potential areas for improving customer retention and profitability. Your goal is to identify customers with a high policy claim amount and a low customer lifetime value.

In the Pandas Lab, we only looked at high policy claim amounts because we couldn't look into low customer lifetime values. If we had tried to work with that column, we wouldn't have been able to because customer lifetime value wasn't clean and in its proper format. So after cleaning and formatting the data, let's get some more interesting insights!

Instructions:

- Review the statistics again for total claim amount and customer lifetime value to gain an understanding of the data.
- To identify potential areas for improving customer retention and profitability, we want to focus on customers with a high policy claim amount and a low customer lifetime value. Consider customers with a high policy claim amount to be those in the top 25% of the total claim amount, and clients with a low customer lifetime value to be those in the bottom 25% of the customer lifetime value. Create a pandas DataFrame object that contains information about customers with a policy claim amount greater than the 75th percentile and a customer lifetime value in the bottom 25th percentile.
- Use DataFrame methods to calculate summary statistics about the high policy claim amount and low customer lifetime value data. To do so, select both columns of the dataframe simultaneously and pass it to the `.describe()` method. This will give you descriptive statistics, such as mean, median, standard deviation, minimum and maximum values for both columns at the same time, allowing you to compare and analyze their characteristics.

In [ ]:
# Your code here